In [ ]:
# default_exp paper.cnn.validation_curve

# 4.2. Validation curve (CNN)

> Computing the validation curve of the CNN for the prediction of exchangeable potassium (K ex.)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Python utils
import math
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y
from mirzai.training.cnn import (Model, weights_init)
from mirzai.data.torch import DataLoaders, SNV_transform
from mirzai.training.cnn import Learner

from fastcore.transform import compose

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split

# Deep Learning stack
import torch
from torch.nn import MSELoss
from torch.optim import Adam
from torch.optim.lr_scheduler import CyclicLR

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
# Is a GPU available?
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
print(f'Runtime is: {device}')

Runtime is: cpu


In [ ]:
n_epochs = 100
criterion = MSELoss() # Mean Squared Error loss
step_size_up = 5
base_lr, max_lr = 3e-5, 1e-3 # Based on Learning Rate
seeds = range(20)

# If no GPU then just for test
if device.type == 'cpu':
    n_epochs = 2
    seeds = range(2)
    X, y = X[:100, :], y[:100]
    
history = OrderedDict({'nb_epochs': range(1, n_epochs + 1), 'train_score': [], 'valid_score': []})

### Run

In [ ]:
for seed in tqdm(seeds):
    print(100*'-')
    print('Seed: {}'.format(seed))

    
    data = train_test_split(X, y, test_size=0.1, random_state=seed)
    X_train, X_test, y_train, y_test = data
    dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
    training_generator, validation_generator = dls.loaders()
    
    model = Model(X_train.shape[1], out_channel=16).to(device)
    opt = Adam(model.parameters(), lr=1e-4)
    model = model.apply(weights_init)

    scheduler = CyclicLR(opt, base_lr=base_lr, max_lr=max_lr,
                         step_size_up=step_size_up, mode='triangular',
                         cycle_momentum=False)
    
  
    learner = Learner(model, criterion, opt, n_epochs=n_epochs, 
                      scheduler=scheduler, verbose=True)
    model, losses = learner.fit(training_generator, validation_generator)        
   
    history['train_score'].append(losses['train'])
    history['valid_score'].append(losses['valid'])

  0%|          | 0/2 [00:00<?, ?it/s]

----------------------------------------------------------------------------------------------------
Seed: 0
Runtime is: cpu


  0%|          | 0/2 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.2089298516511917
 Validation loss: 0.1357429027557373
End of Epoch 2
 Training loss: 0.20794534186522165
 Validation loss: 0.13419437408447266
----------------------------------------------------------------------------------------------------
Seed: 1
Runtime is: cpu


  0%|          | 0/2 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.22680779794851938
 Validation loss: 0.19873850047588348
End of Epoch 2
 Training loss: 0.22417877614498138
 Validation loss: 0.19632939994335175


In [ ]:
history

OrderedDict([('nb_epochs', range(1, 3)),
             ('train_score',
              [[0.19046153128147125, 0.1879679560661316],
               [0.22974275052547455, 0.22835115094979605]]),
             ('valid_score',
              [[0.11871500313282013, 0.11751694977283478],
               [0.19508977234363556, 0.19338101148605347]])])

## Save

In [ ]:
MONITORING_PATH = Path('nameofyourfolder')
with open(MONITORING_PATH/'history_cnn_validation_curve.pickle'), 'wb') as f: 
    pickle.dump(history, f)